In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
input_data_folder = r'C:\Users\david.hensle\OneDrive - Resource Systems Group, Inc\Documents\projects\ODOT\ctramp_to_asim_data_conversion\PopulationSim_to_RSG\HH_GQ_combined'
output_data_folder = r'.\..\metro_data'

assert os.path.exists(input_data_folder), f"Input data folder {input_data_folder} does not exist."
assert os.path.exists(output_data_folder), f"Output data folder {output_data_folder} does not exist."

## Households Data Processing

In [ ]:
in_households = pd.read_csv(os.path.join(input_data_folder, 'households_all.csv'))

hh_renaming_dict = {
    'hhid': 'household_id',
    # 'serialno': 'SERIALNO',  # (Optional, excluding for now)
    'maz': 'MAZ',
    'taz': 'TAZ',
    'htype': 'TYPE',
    'np': 'NP',
    'hht': 'HHT',
    'hhincadj': 'HHINCADJ',
    'nwrkrs_esr': 'WORKERS'
}

out_households = in_households[hh_renaming_dict.keys()].copy()
out_households.rename(columns=hh_renaming_dict, inplace=True)

out_households.head()

In [ ]:
out_households.to_csv(os.path.join(output_data_folder, 'households.csv'), index=False)

## Persons Data Processing

In [ ]:
# Import Persons Table
in_persons = pd.read_csv(os.path.join(input_data_folder, 'persons_all.csv'))

renaming_dict = {
    'hhid': 'household_id',
    'sporder': 'PERSON_NUM',
    'PERID': 'person_id',
    'PUMA': 'PUMA_GEOID',
    # 'taz': 'TAZ',  # optional, excluding for now
    # 'maz': 'MAZ',  # optional, excluding for now
    'agep': 'AGEP',
    'sex': 'SEX',
    'wkhp': 'WKHP',
    'wkw': 'WKW',
    'esr': 'ESR',
    'schg': 'SCHG',
    'mil': 'MIL',
    'occp': 'OCCCAT'
}

final_order = ['household_id','PERSON_NUM','person_id','PUMA_GEOID','AGEP','SEX','WKHP','WKW','ESR','SCHG','MIL','NAICSP','INDP',
'OCCP','OCCCAT','DDRS','DEAR','DEYE','DOUT','DPHY','DREM','TOLLFACTOR','FAREFACTOR']

# Grab data and rename columns
out_persons = in_persons[renaming_dict.keys()].copy()

out_persons.rename(columns=renaming_dict, inplace=True)

# fill in missing columns with empty strings
missing_cols = set(final_order) - set(out_persons.columns)
print(f"Filling missing columns in out_persons with empty string: {missing_cols}")

# Add new column headers to ActivitySim
out_persons[list(missing_cols)] = ''

# Reorder fields again
out_persons = out_persons[final_order]

out_persons.head()


In [ ]:
out_persons.to_csv(os.path.join(output_data_folder, 'persons.csv'), index=False)

## Land Use Processing

In [ ]:
landuse = pd.read_csv(os.path.join(input_data_folder, 'CTRAMP_land_use.csv'))

# CT_RampFields to ActivitySimFields mapping taken from MAZ_Transformation_Walkacross.xlsx
landuse_rename = {
    'MAZ': 'MAZ',
    'TAZ': 'TAZ',
    'BG': 'BLKGRP',
    'TRACT': 'TRACT',
    'PUMA': 'PUMA',
    'COUNTY': 'COUNTY',
    # '--': 'DISTRICTXX',  # No direct source, create empty or calculated if needed
    # The following are calculated fields:
    # 'HH_SF+HH_MF+HH_MH+HH_DUPLEX': 'TOTNGQHHS',
    # 'HH_GQ_CIV+HH_GQ_MIL+HH_GQ_UNIV': 'TOTGQHHS',
    'HH': 'TOTHHS',
    'POP': 'TOTPOP',
    'EMP_AMF': 'EMP_NRM',
    'EMP_CON': 'EMP_CON',
    'EMP_MFG': 'EMP_NHTMFG',
    'EMP_MHT': 'EMP_HTMFG',
    'EMP_WT': 'EMP_WT',
    'EMP_TWU': 'EMP_TWU',
    'EMP_RCS': 'EMP_RET',
    'EMP_PBS': 'EMP_IFRPBS',
    'EMP_HSS': 'EMP_HCS',
    'EMP_OSV': 'EMP_OSV',
    'EMP_GOV': 'EMP_GOV',
    'EMP_EDU': 'EMP_EDU',
    'EMP_AER': 'EMP_AER',
    # '0': 'EMP_ACC',  # Set to 0
    'EMP_FSD': 'EMP_FSD',
    'EMP_TOTAL': 'EMP_TOT',
    'ENROLLK_8': 'ENROLL_K8',
    'ENROLL9_12': 'ENROLL_912',
    'ENROLLCOLL': 'ENROLL_COLL',
    # 'TERMTIME (set to 0)': 'TERMTIME',  # Set to 0
    # 'HPARKCOST*100 (convert to cents)': 'PRKCST_HR',  # Needs conversion
    # 'DPARKCOST*100 (convert to cents)': 'PRKCST_DAY',  # Needs conversion
    # 'MPARKCOST*100*22 (convert to cents and to monthly from daily)': 'PRKCST_MNTH',  # Needs conversion
    # 'MAX(HSTALLSSAM+DSTALLSSAM+MSTALLSSAM+HSTALLSOTH+DSTALLSOTH+MSTALLSOTH)': 'PRKSPACES',  # Needs calculation
    # 'calculated from shapefile': 'ACRES',  # Needs calculation
    'PARKACRES': 'ACTIVE_ACRES',
    # '0': 'OSPC_ACRES',  # Set to 0
    # 'NOT AVAILABLE IN CTRAMP': 'ESCOOACCTIME',  # Set to NaN or 0
    # 'NOT AVAILABLE IN CTRAMP': 'EBIKEACCTIME',  # Set to NaN or 0
    # '0': 'EXTERNAL',  # Set to 0
    # '0': 'EXT_WORK_SIZE',  # Set to 0
    # '0': 'EXT_NWRK_SIZE',  # Set to 0
    # '--': 'TOTINT',  # Set to NaN or 0
    # '--': 'EMPDEN',
    # '--': 'RETDEN',
    # '--': 'HHDEN',
    # '--': 'POPDEN',
    # '--': 'POPEMPDEN',
    # '--': 'WLKTIME_BUS',
    # '--': 'WLKTIME_BRT',
    # '--': 'WLKTIME_LRT',
    # '--': 'WLKTIME_CR',
    # '--': 'EXPPRK_MNTH',
    # '--': 'EXPPRK_DAY',
    # '--': 'EXPPRK_HR',
    'PARKAREA': 'PARKAREA',
    # '--': 'Built Form (not sure on naming)'
}

# Rename columns
landuse = landuse.rename(columns=landuse_rename)

# Create/calculated fields
landuse['TOTNGQHHS'] = landuse[['HH_SF', 'HH_MF', 'HH_MH', 'HH_DUPLEX']].sum(axis=1)
landuse['TOTGQHHS'] = landuse[['HH_GQ_CIV', 'HH_GQ_MIL', 'HH_GQ_UNIV']].sum(axis=1)
landuse['PRKCST_HR'] = landuse['HPARKCOST'] * 100  # converting to cents
landuse['PRKCST_DAY'] = landuse['DPARKCOST'] * 100
landuse['PRKCST_MNTH'] = landuse['MPARKCOST'] * 100 * 22
landuse['PRKSPACES'] = landuse[['HSTALLSSAM', 'DSTALLSSAM', 'MSTALLSSAM', 'HSTALLSOTH', 'DSTALLSOTH', 'MSTALLSOTH']].max(axis=1)

# FIXME the following fields need to be calculated
landuse['EMP_ACC'] = 0  # Accomodations employment not available in CTRAMP
landuse['TERMTIME'] = 0 # terminal time
landuse['ACRES'] = 1  # Placeholder, needs actual calculation from shapefile
landuse['OSPC_ACRES'] = 0  # Open space acres
landuse['ESCOOACCTIME'] = 0  # need calculation for e-scooter and e-bike access times
landuse['EBIKEACCTIME'] = 0
landuse['EXTERNAL'] = 0  # no external zones in landuse file? -- needs to be added
landuse['EXT_WORK_SIZE'] = 0  # total resident work tours from SWIM at external station
landuse['EXT_NWRK_SIZE'] = 0  # non-work resident tours from SWIM at external station

# If you need to add empty columns for fields not available in CT_RampFields:
for col in ['DISTRICTXX','TOTINT','EMPDEN','RETDEN','HHDEN','POPDEN','POPEMPDEN','WLKTIME_BUS','WLKTIME_BRT','WLKTIME_LRT','WLKTIME_CR','EXPPRK_MNTH','EXPPRK_DAY','EXPPRK_HR','Built Form']:
    if col not in landuse.columns:
        landuse[col] = ''

# PARKAREA recoding:
# 1 -> 1, 2/3 -> 2, 4 -> 3
# 1: Constrained parking area (destinations in area have an expected cost and parking location choice model is applied. 
# 2: Buffer area (free parking in area is included in expected cost for destinations in PARKAREA 1, but destinations in area do not have an expected cost and parking location choice model not applied), 
# 3: Not parking constrained or buffer area (free unconstrained parking).  
landuse['PARKAREA'] = landuse['PARKAREA'].replace({2: 2, 3: 2, 4: 3})

# Final output columns for ActivitySim
activitysim_fields = [
    'MAZ','TAZ','BLKGRP','TRACT','PUMA','COUNTY','DISTRICTXX','TOTNGQHHS','TOTGQHHS','TOTHHS','TOTPOP',
    'EMP_NRM','EMP_CON','EMP_NHTMFG','EMP_HTMFG','EMP_WT','EMP_TWU','EMP_RET','EMP_IFRPBS','EMP_HCS','EMP_OSV',
    'EMP_GOV','EMP_EDU','EMP_AER','EMP_ACC','EMP_FSD','EMP_TOT','ENROLL_K8','ENROLL_912','ENROLL_COLL','TERMTIME',
    'PRKCST_HR','PRKCST_DAY','PRKCST_MNTH','PRKSPACES','ACRES','ACTIVE_ACRES','OSPC_ACRES','ESCOOACCTIME','EBIKEACCTIME',
    'EXTERNAL','EXT_WORK_SIZE','EXT_NWRK_SIZE','TOTINT','EMPDEN','RETDEN','HHDEN','POPDEN','POPEMPDEN','WLKTIME_BUS',
    'WLKTIME_BRT','WLKTIME_LRT','WLKTIME_CR','EXPPRK_MNTH','EXPPRK_DAY','EXPPRK_HR','PARKAREA','Built Form'
]
landuse = landuse.reindex(columns=activitysim_fields)

landuse.head()

In [ ]:
# FIXME additional columns needed for current SANDAG implementation
# expect these to be removed or changed in future versions
landuse['walk_dist_local_bus'] = 0.25
landuse['micro_dist_local_bus'] = 0.25
landuse['walk_dist_premium_transit'] = 0.5
landuse['micro_dist_premium_transit'] = 0.5
landuse['MicroAccessTime'] = 0.5
landuse['nev'] = 0
landuse['microtransit'] = 0
landuse['external_work'] = np.where(landuse['TAZ'] <= 10, 10, 0)  # adding non-zero size for external models so model won't crash
landuse['external_nonwork'] = np.where(landuse['TAZ'] <= 10, 10, 0)  # adding non-zero size for external models so model won't crash
landuse['external_MAZ'] = np.where(landuse['TAZ'] <= 10, 1, 0)
landuse['totint'] = 10  # total intersections
# number of park-and-ride spaces in the zone
# just sample a random 10 percent of zones to have park-and-ride spaces
landuse['park_and_ride_spaces'] = np.where(np.random.rand(len(landuse)) < 0.1, np.random.randint(50, 200), 0)

In [ ]:
# Save to CSV
landuse.to_csv(os.path.join(output_data_folder, 'land_use.csv'), index=False)